# Phase 3: SAE basics

Goal: load a pretrained SAE on GPT-2-small, extract activations, find what
each feature responds to. By the end of this notebook I can pick a feature
index and articulate what it represents.

Why this matters for the project: in Phase 4 we'll match features across
two SAEs using GW. To know whether the matches are "good," we need to be
able to look at a matched pair and see whether they fire on similar things.

In [ ]:
from __future__ import annotations

import torch
from sae_lens import SAE, HookedSAETransformer

# Force CPU. Other projects are using the GPU, and 4GB isn't enough to share.
device = "cpu"
print(f"device: {device}")

In [ ]:
# GPT-2 small with sae_lens hooks. ~500MB download first time, then cached.
model = HookedSAETransformer.from_pretrained("gpt2", device=device)

# Pretrained residual-stream SAE on layer 8.
# Use the new API that returns just the SAE (sae_lens 6.x).
sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.8.hook_resid_pre",
    device=device,
)

# Hook name: try the new location, fall back, hard-code if needed.
if hasattr(sae.cfg, "metadata") and hasattr(sae.cfg.metadata, "hook_name"):
    HOOK_NAME = sae.cfg.metadata.hook_name
elif hasattr(sae.cfg, "hook_name"):
    HOOK_NAME = sae.cfg.hook_name
else:
    # Last resort: we know what we asked for.
    HOOK_NAME = "blocks.8.hook_resid_pre"

print(f"SAE input dim (model d_resid): {sae.cfg.d_in}")
print(f"SAE feature dim (dictionary size): {sae.cfg.d_sae}")
print(f"Hook point: {HOOK_NAME}")

In [ ]:
# A small corpus to find what features fire on.
texts = [
    "The cat sat on the mat.",
    "Python is a programming language used for data science.",
    "Once upon a time in a faraway kingdom there lived a princess.",
    "The function returns the sum of two integers.",
    "She walked through the park on a sunny afternoon.",
    "The Roman Empire fell in 476 CE.",
    "Photosynthesis converts sunlight into chemical energy.",
    "He felt a wave of sadness wash over him.",
    "import numpy as np",
    "The quarterback threw a 50-yard touchdown pass.",
]

# Tokenize and run the model, capturing activations at the SAE's hook point.
all_tokens = []
all_activations = []

for text in texts:
    tokens = model.to_tokens(text)  # (1, n_tokens)
    __, cache = model.run_with_cache(tokens, names_filter=[HOOK_NAME])
    activations = cache[HOOK_NAME]
    all_tokens.append(tokens[0])
    all_activations.append(activations[0])  # (n_tokens, d_resid)

# Concatenate so each row is one token's residual-stream activation.
flat_tokens = torch.cat(all_tokens)
flat_activations = torch.cat(all_activations)
print(f"Total tokens: {flat_tokens.shape[0]}")
print(f"Activation tensor shape: {flat_activations.shape}")

In [ ]:
# SAE encodes activations into sparse feature activations.
with torch.no_grad():
    sae_features = sae.encode(flat_activations)  # (N, 24576)

print(f"SAE feature tensor shape: {sae_features.shape}")
print(
    f"Mean fraction of features active per token: "
    f"{(sae_features > 0).float().mean().item():.4f}"
)
print(
    f"Mean number of active features per token: "
    f"{(sae_features > 0).float().sum(dim=1).mean().item():.1f}"
)

In [ ]:
def show_top_activating_tokens(feature_idx: int, k: int = 10) -> None:
    """For a given SAE feature, print the top-k tokens that activate it."""
    feature_acts = sae_features[:, feature_idx]  # (N,)

    # Top-k tokens by activation strength.
    top_vals, top_idxs = torch.topk(feature_acts, k=min(k, len(feature_acts)))

    print(f"Feature {feature_idx} — top {k} activating tokens:")
    for val, idx in zip(top_vals, top_idxs, strict=False):
        if val.item() <= 0:
            break
        token_str = model.to_string(flat_tokens[idx].unsqueeze(0))
        # Show some context: 5 tokens before and after.
        start = max(0, idx.item() - 5)
        end = min(len(flat_tokens), idx.item() + 6)
        context = model.to_string(flat_tokens[start:end])
        print(f"  act={val.item():.3f}  token={token_str!r}  context={context!r}")
    print()


# Try a few random features.
import random

random.seed(0)
sample_features = random.sample(range(sae.cfg.d_sae), 5)
for f in sample_features:
    show_top_activating_tokens(f, k=5)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True,
)

big_tokens = []
big_activations = []
n_docs = 30
max_tokens_per_doc = 64

for i, item in enumerate(dataset):
    if i >= n_docs:
        break
    text = item["text"][:800]
    tokens = model.to_tokens(text)[:, :max_tokens_per_doc]
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens, names_filter=[HOOK_NAME])
    big_tokens.append(tokens[0])
    big_activations.append(cache[HOOK_NAME][0])
    del cache
    if i % 10 == 0:
        print(f"  processed {i+1}/{n_docs} docs")

flat_tokens = torch.cat(big_tokens)
flat_activations = torch.cat(big_activations)

with torch.no_grad():
    sae_features = sae.encode(flat_activations)

print(f"Big corpus tokens: {flat_tokens.shape[0]}")
print(f"SAE features shape: {sae_features.shape}")
print(
    f"Mean active features per token: " f"{(sae_features > 0).float().sum(dim=1).mean().item():.1f}"
)

In [ ]:
# Find features that actually fire on this corpus.
firing_counts = (sae_features > 0).float().sum(dim=0)  # (24576,)
print(f"Features with at least one firing: " f"{(firing_counts > 0).sum().item()}/{sae.cfg.d_sae}")
print(f"Features with at least 10 firings: " f"{(firing_counts >= 10).sum().item()}")

# Inspect the most-frequently-firing features.
top_active_features = torch.topk(firing_counts, k=10).indices
print("\nMost-frequently-firing features:")
for f in top_active_features:
    show_top_activating_tokens(f.item(), k=8)

In [ ]:
# Look for medium-frequency features (fire 30-150 times) and inspect a sample.
import torch as _torch

freq = (sae_features > 0).float().sum(dim=0)
mid_freq_mask = (freq >= 30) & (freq <= 150)
mid_freq_indices = _torch.where(mid_freq_mask)[0]
print(f"Medium-frequency features (30-150 firings): {len(mid_freq_indices)}")

# Sort by max activation strength — features with high peaks tend to be more specific.
max_acts = sae_features.max(dim=0).values
mid_freq_max_acts = max_acts[mid_freq_indices]
top_specific = mid_freq_indices[_torch.topk(mid_freq_max_acts, k=8).indices]

print("\nMost-specific medium-frequency features:")
for f in top_specific:
    show_top_activating_tokens(f.item(), k=10)